# Ensemble Learning: AdaBoost Regression on the Boston Housing Dataset

---

## Introduction

**AdaBoost** (Adaptive Boosting) for regression — implemented as `AdaBoostRegressor` in scikit-learn — works on the same sequential principle as its classification counterpart, but adapts to continuous targets:

- Each weak learner is a shallow `DecisionTreeRegressor` (default `max_depth=3`)
- After each round, sample weights are updated based on prediction **error magnitude** — samples with larger errors receive higher weights in the next round
- The final prediction is a **weighted median** (or weighted mean) of all weak learner predictions

### AdaBoost Regression vs. Classification

| Aspect | AdaBoost Classification | AdaBoost Regression |
|---|---|---|
| Target | Discrete labels | Continuous values |
| Error measure | Misclassification | Absolute / square / exponential loss |
| Weight update | Based on correct/wrong | Based on normalized error magnitude |
| Final output | Weighted majority vote | Weighted median of predictions |
| sklearn class | `AdaBoostClassifier` | `AdaBoostRegressor` |

### Workflow

1. Load and explore the Boston Housing dataset
2. Preprocess and split into train / test sets
3. Establish a baseline with a single `DecisionTreeRegressor`
4. Train `AdaBoostRegressor` and evaluate with R² and RMSE
5. Analyze the effect of `n_estimators` and `learning_rate`
6. Tune hyperparameters with `GridSearchCV`
7. Visualize feature importances and predicted vs. actual
8. Compare all models in a results summary

---

## 1. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

---

## 2. Loading and Exploring the Dataset

The **Boston Housing dataset** contains 506 samples describing housing attributes in Boston suburbs. The target variable `medv` is the median home value in thousands of dollars.

| Feature | Description |
|---|---|
| `crim` | Per-capita crime rate |
| `zn` | Proportion of residential land for large lots |
| `indus` | Proportion of non-retail business acres |
| `chas` | Charles River dummy variable |
| `nox` | Nitric oxide concentration |
| `rm` | Average rooms per dwelling |
| `age` | Proportion of units built before 1940 |
| `dis` | Distance to employment centres |
| `rad` | Highway accessibility index |
| `tax` | Property tax rate |
| `ptratio` | Pupil-teacher ratio |
| `b` | Proportion of residents by town |
| `lstat` | Percentage lower-status population |
| `medv` | **Target**: Median home value ($1,000s) |

In [ ]:
df = pd.read_csv('BostonHousing2.csv')

print('Shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
df.head()

In [ ]:
df.describe().round(2)

### 2.1 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['medv'], bins=30, color='steelblue', edgecolor='k', linewidth=0.4)
axes[0].set_xlabel('medv ($1,000s)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution — medv')

corr = df.corr()['medv'].drop('medv').sort_values()
corr.plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='k', linewidth=0.4)
axes[1].set_title('Feature Correlation with Target (medv)')
axes[1].set_xlabel('Pearson Correlation')

plt.tight_layout()
plt.show()

---

## 3. Preprocessing and Train / Test Split

In [ ]:
X = df.drop('medv', axis=1)
y = df['medv']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features — AdaBoost with trees does not strictly require scaling,
# but it ensures fair comparison with other estimators
scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print(f'Features : {X_train.shape[1]}')

---

## 4. Baseline: Single Decision Tree

An unconstrained `DecisionTreeRegressor` fits the training data perfectly but overfits badly — the large train/test gap in R² confirms high variance.

In [ ]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_sc, y_train)

y_pred_dt   = dt.predict(X_test_sc)
r2_dt       = r2_score(y_test, y_pred_dt)
rmse_dt     = mean_squared_error(y_test, y_pred_dt, squared=False)
r2_train_dt = r2_score(y_train, dt.predict(X_train_sc))

print(f'Decision Tree  —  Train R²: {r2_train_dt:.4f}   Test R²: {r2_dt:.4f}   RMSE: {rmse_dt:.4f}')

---

## 5. AdaBoost Regressor

### 5.1 Training

`AdaBoostRegressor` uses a `DecisionTreeRegressor` with `max_depth=3` as the default base estimator. The `loss` parameter controls how sample weights are updated — `'linear'` (default), `'square'`, or `'exponential'` error scaling.

In [ ]:
ada = AdaBoostRegressor(
    estimator=DecisionTreeRegressor(max_depth=3),
    n_estimators=100,
    learning_rate=0.1,
    loss='linear',
    random_state=42
)

ada.fit(X_train_sc, y_train)

y_pred_ada   = ada.predict(X_test_sc)
r2_ada       = r2_score(y_test, y_pred_ada)
rmse_ada     = mean_squared_error(y_test, y_pred_ada, squared=False)
r2_train_ada = r2_score(y_train, ada.predict(X_train_sc))

print(f'AdaBoost  —  Train R²: {r2_train_ada:.4f}   Test R²: {r2_ada:.4f}   RMSE: {rmse_ada:.4f}')

### 5.2 Predicted vs. Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, y_pred, title, r2 in zip(
    axes,
    [y_pred_dt, y_pred_ada],
    ['Decision Tree', 'AdaBoost (n=100)'],
    [r2_dt, r2_ada]
):
    ax.scatter(y_test, y_pred, alpha=0.6, edgecolors='k', linewidths=0.3, s=30)
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual medv')
    ax.set_ylabel('Predicted medv')
    ax.set_title(f'{title}  |  R² = {r2:.4f}')
    ax.legend(fontsize=9)

plt.suptitle('Predicted vs. Actual — Decision Tree vs. AdaBoost', fontsize=13)
plt.tight_layout()
plt.show()

---

## 6. Effect of n_estimators and Learning Rate

The `learning_rate` shrinks the contribution of each weak learner. A lower rate requires more estimators but often generalizes better. We plot test R² across a range of `n_estimators` for three different learning rates.

In [ ]:
estimator_range = range(10, 301, 10)
learning_rates  = [0.01, 0.1, 1.0]

plt.figure(figsize=(10, 5))

for lr in learning_rates:
    scores = []
    for n in estimator_range:
        model = AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3),
            n_estimators=n, learning_rate=lr,
            random_state=42
        )
        model.fit(X_train_sc, y_train)
        scores.append(r2_score(y_test, model.predict(X_test_sc)))
    plt.plot(estimator_range, scores, lw=2, label=f'lr = {lr}')

plt.xlabel('Number of Estimators')
plt.ylabel('Test R²')
plt.title('AdaBoost — Test R² vs. n_estimators by Learning Rate')
plt.legend()
plt.tight_layout()
plt.show()

---

## 7. Hyperparameter Tuning with GridSearchCV

| Parameter | Description |
|---|---|
| `n_estimators` | Number of boosting rounds |
| `learning_rate` | Shrinkage factor per estimator |
| `loss` | Error function for weight updates |
| `estimator__max_depth` | Depth of each base decision tree |

In [ ]:
%%time

param_grid = {
    'n_estimators':          [50, 100, 200],
    'learning_rate':         [0.01, 0.05, 0.1, 0.5],
    'loss':                  ['linear', 'square', 'exponential'],
    'estimator__max_depth':  [2, 3, 4]
}

grid_search = GridSearchCV(
    AdaBoostRegressor(estimator=DecisionTreeRegressor(), random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_sc, y_train)

In [ ]:
best = grid_search.best_estimator_
y_pred_tuned = best.predict(X_test_sc)
r2_tuned     = r2_score(y_test, y_pred_tuned)
rmse_tuned   = mean_squared_error(y_test, y_pred_tuned, squared=False)

print('Grid Search Results')
print('-' * 50)
print(f'CV R² (best)   : {grid_search.best_score_:.4f}')
print(f'Test R²        : {r2_tuned:.4f}')
print(f'Test RMSE      : {rmse_tuned:.4f}')
print(f'\nBest Parameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k:<30} {v}')

---

## 8. Feature Importances

AdaBoost inherits feature importances from its base estimators — the values reflect the mean decrease in impurity across all weak learners, weighted by their contribution to the ensemble.

In [ ]:
importances = pd.Series(
    ada.feature_importances_,
    index=X.columns
).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', edgecolor='k', linewidth=0.4)
plt.xlabel('Feature Importance')
plt.title('AdaBoost — Feature Importances')
plt.tight_layout()
plt.show()

print('\nFeature Importances (ranked):')
print(importances.sort_values(ascending=False).round(4).to_string())

---

## 9. Results Summary

In [ ]:
cv_dt  = cross_val_score(DecisionTreeRegressor(random_state=42),
                         X_train_sc, y_train, cv=5, scoring='r2').mean()
cv_ada = cross_val_score(AdaBoostRegressor(
                             estimator=DecisionTreeRegressor(max_depth=3),
                             n_estimators=100, learning_rate=0.1, random_state=42),
                         X_train_sc, y_train, cv=5, scoring='r2').mean()
cv_tuned = cross_val_score(best, X_train_sc, y_train, cv=5, scoring='r2').mean()

results = pd.DataFrame({
    'Model': [
        'Decision Tree (baseline)',
        'AdaBoost (n=100, lr=0.1)',
        'AdaBoost (GridSearchCV)'
    ],
    'Test R²':  [round(r2_dt, 4),    round(r2_ada, 4),    round(r2_tuned, 4)],
    'Test RMSE':[round(rmse_dt, 4),  round(rmse_ada, 4),  round(rmse_tuned, 4)],
    'CV R²':    [round(cv_dt, 4),    round(cv_ada, 4),    round(cv_tuned, 4)]
})

results = results.sort_values('Test R²', ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

---

## Conclusion

This notebook applied `AdaBoostRegressor` to the Boston Housing dataset across a complete regression workflow.

**Key findings:**

- The single **Decision Tree** achieves a perfect train R² of 1.0 but drops significantly on the test set — the hallmark of high variance and overfitting. The predicted vs. actual scatter plot shows poor alignment with the diagonal.
- **AdaBoost** with 100 estimators and `learning_rate=0.1` substantially improves test R² and RMSE over the baseline. By sequentially focusing each new tree on the hardest residuals, the ensemble captures non-linear patterns that a single tree overfits.
- The **learning rate plot** confirms the classic boosting trade-off: `lr=1.0` converges quickly but can overfit with many estimators; `lr=0.01` is slow but stable; `lr=0.1` is a reliable default that balances both.
- **GridSearchCV** identifies the optimal combination of `n_estimators`, `learning_rate`, `loss`, and base tree depth, further improving test R² with proper regularization.
- **Feature importances** show `lstat` (lower-status population %) and `rm` (average rooms) as the dominant predictors — consistent with their strong correlations to `medv` visible in the correlation bar chart.

**Takeaways:**

- `learning_rate` and `n_estimators` are inversely related — halving the learning rate typically requires doubling the estimators for equivalent performance. The learning rate plot makes this trade-off directly visible.
- The `loss` parameter is unique to `AdaBoostRegressor`: `'square'` and `'exponential'` penalize large errors more aggressively than `'linear'`, which can help when the target has outliers.
- AdaBoost is more sensitive to noisy labels than Random Forest — samples with large irreducible errors accumulate very high weights and can destabilize later rounds.
- The natural next step is **Gradient Boosting**, which generalizes AdaBoost by fitting each new tree directly to the residuals (negative gradient of the loss), enabling more flexible loss functions and better regularization.